# Predictive Alerting for Cloud Metrics — Exploration

This notebook demonstrates the full pipeline:
1. Synthetic data generation
2. Feature engineering
3. Model training (Ensemble: IsolationForest + XGBoost)
4. Evaluation (recall, lead time, PR curve)
5. Alert simulation

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import precision_recall_curve

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 4)

## 1. Generate Synthetic Data

In [ ]:
from src.data.generator import generate_dataset

df = generate_dataset(n_days=90, freq_minutes=1, incident_rate=0.02, seed=42)
print(f'Dataset shape: {df.shape}')
print(f'Incident rate: {df["incident"].mean():.3%}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
df.head()

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 10))
metric_cols = ['cpu_utilization', 'memory_usage', 'request_latency', 'error_rate', 'network_in', 'network_out']
sample = df.iloc[:2880]

for ax, col in zip(axes.flat, metric_cols):
    ax.plot(sample.index, sample[col], linewidth=0.7, alpha=0.8)
    incident_mask = sample['incident']
    ax.fill_between(sample.index, sample[col].min(), sample[col].max(),
                    where=incident_mask, alpha=0.3, color='red', label='Incident')
    ax.set_title(col.replace('_', ' ').title())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))

handles = [plt.Rectangle((0, 0), 1, 1, fc='red', alpha=0.3)]
fig.legend(handles, ['Incident period'], loc='upper right')
fig.suptitle('Cloud Metrics — First 2 Days (with incident highlights)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
from src.data.preprocessing import create_features, create_targets, split_temporal

features = create_features(df, windows=[5, 15, 30, 60], lags=[1, 5, 15, 30])
targets = create_targets(df.loc[features.index], lookahead_minutes=15)

features = features.loc[targets.index]
targets = targets.loc[features.index]

print(f'Feature matrix shape: {features.shape}')
print(f'Target positive rate: {targets.mean():.3%}')
print(f'\nFeature groups:')
for prefix in ['_mean_', '_std_', '_lag_', '_roc_', '_q75_', 'hour_', 'dow_']:
    count = sum(1 for c in features.columns if prefix in c)
    print(f'  {prefix.strip("_"):12s}: {count} features')

In [ ]:
corr_cols = ['cpu_utilization', 'memory_usage', 'request_latency', 'error_rate',
             'cpu_utilization_mean_15m', 'error_rate_roc_1m', 'request_latency_q95_60m']
corr_cols = [c for c in corr_cols if c in features.columns]

combined = features[corr_cols].copy()
combined['target'] = targets.values

plt.figure(figsize=(10, 8))
sns.heatmap(combined.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Train/Test Split and Model Training

In [ ]:
train_features, test_features = split_temporal(features, test_ratio=0.2)
train_targets = targets.loc[train_features.index]
test_targets = targets.loc[test_features.index]

print(f'Train: {len(train_features):,} rows | {train_targets.sum():,} positive ({train_targets.mean():.2%})')
print(f'Test:  {len(test_features):,} rows | {test_targets.sum():,} positive ({test_targets.mean():.2%})')
print(f'Train period: {train_features.index.min().date()} to {train_features.index.max().date()}')
print(f'Test period:  {test_features.index.min().date()} to {test_features.index.max().date()}')

In [ ]:
from src.models.ensemble import EnsembleModel

model = EnsembleModel(
    iso_weight=0.3,
    xgb_weight=0.7,
    iso_params={'n_estimators': 100},
    xgb_params={'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
                'subsample': 0.8, 'colsample_bytree': 0.8},
)

print('Training ensemble model...')
model.fit(train_features, train_targets)
print('Training complete.')

## 4. Evaluation

In [ ]:
from src.evaluation.metrics import full_evaluation_report, find_threshold_for_recall

test_scores = model.predict_proba(test_features)
report = full_evaluation_report(test_targets, test_scores, test_features.index)

print(f'ROC-AUC: {report["roc_auc"]:.4f}')
print(f'Optimal threshold for 80% recall: {report["optimal_threshold_for_recall_0.8"]:.3f}')
opt = report['at_optimal_threshold']
print(f'  Recall:    {opt["recall"]:.3f}')
print(f'  FPR:       {opt["fpr"]:.3f}')
lt = opt['lead_time']
if lt['mean_lead_time']:
    print(f'  Lead time: mean={lt["mean_lead_time"]:.1f}m, median={lt["median_lead_time"]:.1f}m')

In [ ]:
thresholds = sorted(report['threshold_metrics'].keys(), key=float)
recalls = [report['threshold_metrics'][t]['recall'] for t in thresholds]
precisions = [report['threshold_metrics'][t]['precision'] for t in thresholds]
fprs = [report['threshold_metrics'][t]['fpr'] for t in thresholds]

pr_precisions, pr_recalls, pr_thresholds = precision_recall_curve(test_targets, test_scores)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(pr_recalls, pr_precisions, linewidth=2, color='steelblue')
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')

t_vals = [float(t) for t in thresholds]
axes[1].plot(t_vals, recalls, 'o-', label='Recall', color='green')
axes[1].plot(t_vals, fprs, 's-', label='FPR', color='red')
axes[1].axhline(y=0.8, color='green', linestyle='--', alpha=0.4)
axes[1].axvline(x=report['optimal_threshold_for_recall_0.8'], color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Threshold')
axes[1].set_title('Recall and FPR vs Threshold')
axes[1].legend()

axes[2].plot(fprs, recalls, 'D-', color='purple')
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('Recall')
axes[2].set_title('Recall vs FPR Trade-off')
for t, r, f in zip(t_vals, recalls, fprs):
    axes[2].annotate(f'{t:.2f}', (f, r), textcoords='offset points', xytext=(5, 3), fontsize=8)

plt.suptitle('Model Evaluation Metrics', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
threshold = report['optimal_threshold_for_recall_0.8']
sample_test = test_features.iloc[:2880].copy()
sample_scores = model.predict_proba(sample_test)
sample_targets = test_targets.loc[sample_test.index]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

ax1.plot(sample_test.index, sample_scores, linewidth=0.8, color='steelblue', label='Anomaly Score')
ax1.axhline(y=threshold, color='red', linestyle='--', linewidth=1.5, label=f'Threshold ({threshold:.2f})')
ax1.fill_between(sample_test.index, 0, 1, where=sample_targets.values,
                  alpha=0.2, color='red', label='Incident')
ax1.set_ylabel('Score')
ax1.set_title('Ensemble Anomaly Score vs Incident Labels (first 2 days of test set)')
ax1.legend(loc='upper right')
ax1.set_ylim(0, 1)

incident_df = df[['incident']].loc[sample_test.index]
ax2.fill_between(sample_test.index, 0, 1, where=incident_df['incident'].values,
                  color='red', alpha=0.5, label='True Incident')
alerts = sample_scores >= threshold
ax2.fill_between(sample_test.index, 0, 1, where=alerts,
                  color='orange', alpha=0.4, label='Alert Triggered')
ax2.set_ylabel('Active')
ax2.set_xlabel('Time')
ax2.set_title('Alerts vs True Incidents')
ax2.legend(loc='upper right')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
plt.setp(ax2.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

## 5. Lead Time Distribution

In [ ]:
from src.evaluation.metrics import compute_lead_time

lt_result = compute_lead_time(test_features.index, test_targets, test_scores, threshold)

print(f'Incidents with advance warning: {lt_result["n_with_lead_time"]} / {lt_result["n_incidents"]}')
if lt_result['distribution']:
    print(f'Mean lead time:   {lt_result["mean_lead_time"]:.1f} minutes')
    print(f'Median lead time: {lt_result["median_lead_time"]:.1f} minutes')

    plt.figure(figsize=(10, 4))
    plt.hist(lt_result['distribution'], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    plt.axvline(lt_result['mean_lead_time'], color='red', linestyle='--', label=f'Mean: {lt_result["mean_lead_time"]:.1f}m')
    plt.axvline(lt_result['median_lead_time'], color='orange', linestyle='--', label=f'Median: {lt_result["median_lead_time"]:.1f}m')
    plt.xlabel('Lead Time (minutes)')
    plt.ylabel('Count')
    plt.title('Distribution of Alert Lead Times Before Incidents')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 6. Feature Importance

In [ ]:
xgb_model = model.xgb_model._model
importances = pd.Series(
    xgb_model.feature_importances_,
    index=train_features.columns
).sort_values(ascending=False)

top_n = 25
plt.figure(figsize=(12, 7))
importances.head(top_n).sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title(f'Top {top_n} XGBoost Feature Importances')
plt.xlabel('Importance (gain)')
plt.tight_layout()
plt.show()

## 7. Alert Manager Simulation

In [ ]:
from src.alerting.alert_manager import AlertManager

manager = AlertManager(threshold=threshold, cooldown_minutes=5)

alerts_fired = []
for ts, score in zip(test_features.index[:2880], test_scores[:2880]):
    alert = manager.check_and_alert(ts.to_pydatetime(), float(score))
    if alert:
        alerts_fired.append(alert)

print(f'Total alerts fired in first 2 days of test set: {len(alerts_fired)}')
for a in alerts_fired[:5]:
    print(f'  {a.timestamp} — score: {a.score:.3f}')

## 8. Full Training Pipeline

In [ ]:
import yaml
from src.training.pipeline import TrainingPipeline

with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

config['storage']['type'] = 'local'
config['model']['xgboost']['n_estimators'] = 300

pipeline = TrainingPipeline(config)
result = pipeline.run(df)

print(f'Artifact location: {result["artifact_location"]}')
print(f'Train size: {result["train_size"]:,}')
print(f'Test size:  {result["test_size"]:,}')
opt = result['metrics'].get('at_optimal_threshold', {})
print(f'Recall at optimal threshold: {opt.get("recall", "N/A"):.3f}')
print(f'FPR at optimal threshold:    {opt.get("fpr", "N/A"):.3f}')
print(f'ROC-AUC: {result["metrics"].get("roc_auc", "N/A"):.4f}')